# Failure experiments (phone set)

One small inference-only experiment per selected failure photo. Photos, hypotheses, variants and refutation criteria are in `configs/failure_experiments.yaml` and were written before any variant was run.

- Every variant is predicted with the evaluation code of the trained checkpoint. Masks are mapped back to the original photo before matching (score >= 0.25, IoU >= 0.5).
- The unchanged photo must reproduce the exported predictions (check below).
- `top class` in brackets is the highest-scoring prediction of any class with IoU >= 0.5 on the object, also below the score threshold.
- One photo per experiment: a result supports or contradicts an explanation for that photo. It does not measure how often the explanation holds.
- Run time about 10 minutes (three models) on the RTX 4050, GPU needed. `SMOKE = True` runs the crop experiment with YOLO only (about 1 minute) and writes to `runs/smoke/`.

## Parameters

In [ ]:
MODELS = ["yolo", "maskrcnn", "m2f"]
SIZE = 640
OUT_DIR = "results/failures"
SMOKE = False

In [ ]:
%config InlineBackend.figure_formats = ['jpeg']
import logging

import matplotlib.pyplot as plt
import pandas as pd

from iiss.analysis import failure_experiments as fe
from iiss.analysis import failures as fa
from iiss.utils.paths import repo_root

logging.basicConfig(level=logging.WARNING)
pd.set_option("display.max_columns", 30, "display.max_rows", 100, "display.width", 200)
pd.set_option("display.max_colwidth", 90, "display.float_format", "{:.3f}".format)

In [ ]:
config = fe.load_experiments()
specs = config["experiments"]
if SMOKE:
    MODELS, OUT_DIR = ["yolo"], "runs/smoke/failures"
    specs = {k: v for k, v in specs.items() if k in ("crop", "mask_grid")}
runs = fe.default_run_ids({"models": MODELS, "input_size": SIZE})
split, limits = fa.load_gt_set("phone"), fa.default_limits()
print(runs, limits, sep="\n")

## Run all variants

In [ ]:
outcomes = fe.run_experiments(specs, split, runs, limits)
table = fe.result_table(outcomes)
print(len(outcomes), "predictions,", len(table), "target rows")

Baseline check: the unchanged photo compared with the exported predictions. `same_tags` must be True in every row.

In [ ]:
baseline = fe.check_baseline(outcomes, split, limits)
assert baseline["same_tags"].all()
baseline

In [ ]:
def fig_path(name):
    return repo_root() / OUT_DIR / f"exp_{name}.jpg"


def show(name):
    s = specs[name]
    print(s["photo"], "|", s["failure"], "| focus", s["focus"])
    print("Hypothesis:", s["hypothesis"].strip(), "\nRefuted if:", s["refuted_if"])
    if name != "mask_grid":
        for run in runs:
            fe.plot_variants(
                outcomes, name, run, split.category_names, save_to=fig_path(f"{name}_{run}")
            )
            plt.show()

## 1. Low light

Gamma and CLAHE brighten the photo. The chair is the target.

In [ ]:
if "low_light" in specs:
    show("low_light")
    display(fe.overview(table, "low_light"))

## 2. Zoom out

The photo is shrunk on a gray canvas. The canvas itself is new image content and the models may predict walls on it.

In [ ]:
if "zoom_out" in specs:
    show("zoom_out")
    display(fe.overview(table, "zoom_out"))

## 3. Mask grid

No model: the GT mask goes through a G x G grid and back, the best IoU a mask head of that resolution can reach. `iou_pred_box` uses the extent of the predicted mask.

In [ ]:
if "mask_grid" in specs:
    show("mask_grid")
    grid, masks = fe.grid_bound_table(specs["mask_grid"], split, f"maskrcnn_s{SIZE}_seed42", limits)
    fe.plot_grid_bound(specs["mask_grid"], split, grid, masks, fig_path("mask_grid"))
    display(grid)

## 4. Crop

The photo is cropped around the light, so the light gets larger.

In [ ]:
if "crop" in specs:
    show("crop")
    display(fe.overview(table, "crop"))

## 5. Wall edge

CLAHE, a black line on the GT boundary, or a darker second wall.

In [ ]:
if "wall_edge" in specs:
    show("wall_edge")
    display(fe.overview(table, "wall_edge"))

## Save the tables

In [ ]:
fe.write_tables(table, grid, OUT_DIR)
print("written to", OUT_DIR)